# Pr 2 - Data Cleanser 

## Aim
To Practice data preprocessing and features engineering with a strong emphasis on handling missing values and outlier detection/removal.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy.stats import zscore
from scipy.stats.mstats import winsorize
import matplotlib.pyplot as plt


## 2. Read the Dataset

In [2]:
df = pd.read_csv("data/patient_health_records.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()


Rows: 1200
Columns: 9


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,20001,49.0,Male,East,NaN,144.0,173.4,101.2,0
1,20002,23.0,Female,East,28.5,117.8,243.6,127.2,1
2,20003,45.0,Female,East,NaN,121.7,182.3,NaN,0
3,20004,35.0,Male,North,23.4,126.7,205.3,105.7,0
4,20005,46.0,Male,East,27.2,124.5,166.5,83.5,0


## 3. Initial Inspection

In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   patient_id      1200 non-null   int64  
 1   age             1128 non-null   float64
 2   gender          1152 non-null   object 
 3   region          1140 non-null   object 
 4   bmi             1116 non-null   float64
 5   blood_pressure  1200 non-null   float64
 6   cholesterol     1128 non-null   float64
 7   glucose         1116 non-null   float64
 8   disease_risk    1200 non-null   int64  
dtypes: float64(5), int64(2), object(2)
memory usage: 84.5+ KB


## 4. Missing Value Report

In [4]:
missing_report = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Percentage": (df.isna().mean() * 100).round(2)
})
missing_report


,Missing Values,Percentage
patient_id,0,0.0
age,72,6.0
gender,48,4.0
region,60,5.0
bmi,84,7.0
blood_pressure,0,0.0
cholesterol,72,6.0
glucose,84,7.0
disease_risk,0,0.0


## Part A – Missing Value Treatment

In [5]:
numeric_cols = ["age", "bmi", "blood_pressure", "cholesterol", "glucose"]
categorical_cols = ["gender", "region"]


### 5. Mean Imputation

In [6]:
mean_data = df.copy()
imputer = SimpleImputer(strategy="mean")
mean_data[numeric_cols] = imputer.fit_transform(mean_data[numeric_cols])
print(mean_data[numeric_cols].isna().sum())


age               0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
dtype: int64


### 6. Median Imputation

In [7]:
median_data = df.copy()
imputer = SimpleImputer(strategy="median")
median_data[numeric_cols] = imputer.fit_transform(median_data[numeric_cols])
print(median_data[numeric_cols].isna().sum())


age               0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
dtype: int64


### 7. Most Frequent Imputation

In [8]:
mode_data = df.copy()
imputer = SimpleImputer(strategy="most_frequent")
mode_data[categorical_cols] = imputer.fit_transform(mode_data[categorical_cols])
print(mode_data[categorical_cols].isna().sum())


gender    0
region    0
dtype: int64


### 8. Random Sample Imputation + Missing Indicator

In [10]:
random_data = df.copy()
rng = np.random.default_rng(42)

for col in numeric_cols:
    random_data[col + "_missing"] = random_data[col].isna().astype(int)

for col in numeric_cols:
    mask = random_data[col].isna()
    observed = random_data.loc[~mask, col]
    if mask.sum() > 0:
        random_data.loc[mask, col] = rng.choice(observed.to_numpy(), size=mask.sum())

print(random_data[numeric_cols].isna().sum())


age               0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
dtype: int64


### 9. KNN Imputation

In [11]:
knn_data = df.copy()
knn = KNNImputer(n_neighbors=5)
knn_data[numeric_cols] = knn.fit_transform(knn_data[numeric_cols])
print(knn_data[numeric_cols].isna().sum())


age               0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
dtype: int64


### 10. MICE Imputation

In [12]:
mice_data = df.copy()
mice = IterativeImputer(max_iter=10, random_state=42)
mice_data[numeric_cols] = mice.fit_transform(mice_data[numeric_cols])
print(mice_data[numeric_cols].isna().sum())


age               0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
dtype: int64


## Part B – Outlier Analysis

In [14]:
outlier_cols = ["bmi", "blood_pressure", "cholesterol", "glucose"]


### 11. Z-Score

In [15]:
for col in outlier_cols:
    scores = np.abs(zscore(df[col], nan_policy="omit"))
    print(f"{col}: {(scores > 3).sum()} potential outliers")


bmi: 5 potential outliers
blood_pressure: 5 potential outliers
cholesterol: 5 potential outliers
glucose: 5 potential outliers


### 12. IQR

In [16]:
for col in outlier_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {count} potential outliers")


bmi: 7 potential outliers
blood_pressure: 7 potential outliers
cholesterol: 13 potential outliers
glucose: 11 potential outliers


### 13. Percentile Method

In [17]:
for col in outlier_cols:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {count} potential outliers")


bmi: 24 potential outliers
blood_pressure: 24 potential outliers
cholesterol: 24 potential outliers
glucose: 12 potential outliers


### 14. Winsorization

In [18]:
winsorized_data = df.copy()

for col in outlier_cols:
    winsorized_data[col] = winsorize(
        winsorized_data[col],
        limits=[0.01, 0.01]
    )

winsorized_data.head()


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,20001,49.0,Male,East,NaN,144.0,173.4,101.2,0
1,20002,23.0,Female,East,28.5,117.8,243.6,127.2,1
2,20003,45.0,Female,East,NaN,121.7,182.3,NaN,0
3,20004,35.0,Male,North,23.4,126.7,205.3,105.7,0
4,20005,46.0,Male,East,27.2,124.5,166.5,83.5,0


## Part C – Final Dataset

In [19]:
final_df = df.copy()

final_num = SimpleImputer(strategy="median")
final_df[numeric_cols] = final_num.fit_transform(final_df[numeric_cols])

final_cat = SimpleImputer(strategy="most_frequent")
final_df[categorical_cols] = final_cat.fit_transform(final_df[categorical_cols])

for col in outlier_cols:
    final_df[col] = winsorize(final_df[col], limits=[0.01, 0.01])


### 15. Final Check

In [20]:
print("Missing values after cleaning:")
print(final_df.isna().sum())
print("\nFinal shape:", final_df.shape)
final_df.head()


Missing values after cleaning:
patient_id        0
age               0
gender            0
region            0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
disease_risk      0
dtype: int64

Final shape: (1200, 9)


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,20001,49.0,Male,East,25.9,144.0,173.4,101.2,0
1,20002,23.0,Female,East,28.5,117.8,243.6,127.2,1
2,20003,45.0,Female,East,25.9,121.7,182.3,106.8,0
3,20004,35.0,Male,North,23.4,126.7,205.3,105.7,0
4,20005,46.0,Male,East,27.2,124.5,166.5,83.5,0


### 16. Before and After

In [21]:
print("Before cleaning:")
display(df[outlier_cols].describe())

print("After cleaning:")
display(final_df[outlier_cols].describe())


Before cleaning:


,bmi,blood_pressure,cholesterol,glucose
count,1116.000000,1200.000000,1128.000000,1116.000000
mean,25.892742,121.332833,198.915514,107.379211
std,5.056737,15.533958,37.353198,25.455698
min,15.000000,85.000000,100.000000,60.000000
25%,22.500000,111.200000,174.275000,91.100000
50%,25.900000,120.500000,198.300000,106.800000
75%,28.900000,131.500000,221.925000,121.825000
max,60.000000,245.000000,450.000000,350.000000


After cleaning:


,bmi,blood_pressure,cholesterol,glucose
count,1200.000000,1200.000000,1200.000000,1200.000000
mean,25.823583,121.042333,198.502083,106.818917
std,4.571603,14.040868,34.181139,21.787316
min,15.300000,87.600000,114.000000,60.000000
25%,22.800000,111.200000,176.575000,92.275000
50%,25.900000,120.500000,198.300000,106.800000
75%,28.600000,131.500000,220.225000,119.925000
max,37.300000,156.500000,286.800000,167.100000


### 17. Save Result

In [22]:
final_df.to_csv(
    "output/final_cleaned_patient_health_records.csv",
    index=False
)
print("Final cleaned dataset saved.")


Final cleaned dataset saved.
